In [21]:
from pathlib import Path
import polars as pl

data_dir = Path().absolute() / ".." / "data" / "parsed" / "full_takeout"

df = pl.read_parquet(data_dir / "clxenc3fw0007gzz3pdz6enfe.snappy")

In [22]:
daily_dfs = df.with_columns(
    date=pl.col("timestamp").dt.date()
).partition_by("date", as_dict=True, include_key=False)


/var/folders/bg/q288_3h54pg5fgrm5np7xp8h0000gn/T/ipykernel_17742/1448968356.py:1: DeprecationWarning: `partition_by(..., as_dict=True)` will change to always return tuples as dictionary keys. Pass `by` as a list to silence this warning, e.g. `partition_by(['date'], as_dict=True)`.
  daily_dfs = df.with_columns(


In [23]:
chunk_size = 10
chunks={}

for date, day_df in daily_dfs.items():
    chunks[date] = []
    for slice in day_df.iter_slices(chunk_size):
        chunks[date].append(slice.select("hour", "title"))

In [24]:
def df_to_text(df):
    text_output = ""
    for row in df.iter_rows():
        hour, title = row
        text_output += f"{hour}: {title}\n"
    return text_output

In [25]:
import datetime
d = chunks[datetime.date(2023, 8, 17)][0]

print(df_to_text(d))

12:19: Searched for google takeout parser
12:18: Searched for opt in advertising
12:09: Searched for Dataswyft crunchbase
12:02: Viewed DataSwift Network Services Ltd.
12:02: Searched for dataswift
11:50: Visited How to change default terminal application in Gnome-Shell - Ask ...
11:50: Searched for default terminal gnome
11:48: Searched for gnome default te
11:43: Visited People Data Labs - Crunchbase Company Profile & Funding
11:43: Searched for people data labs crunchbase

